# 12 — AG-UI Streaming

**AWS-side equivalent:** AG-UI 16-event taxonomy over SSE on AgentCore Runtime
**GCP service:** Same AG-UI taxonomy, self-hosted on Cloud Run (or Agent Engine for production)

AWS-side notebook: `notebooks/aws/12-ag-ui-streaming.ipynb`

_As of 2026-05: there is no GCP-managed AG-UI server. The protocol layer is the same `ag-ui-protocol` Python package (`ag_ui.core` for event models, `ag_ui.encoder` for SSE wire format). We host it ourselves with FastAPI + a Gemini call; deploy the same handler to Cloud Run when promoting beyond the notebook._


## Steps

1. Build a tiny Gemini-backed handler that yields the AG-UI event sequence (RUN_STARTED → TEXT_MESSAGE_* → TOOL_CALL_* → RUN_FINISHED)
2. Wrap it in FastAPI as a `StreamingResponse(media_type="text/event-stream")`
3. Run uvicorn in a background thread inside the notebook
4. Hit it with a POST + ReadableStream-style client (`httpx.stream`) and print the decoded events


## Step 1: Load config


In [17]:
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
MODEL_ID = gcp.get('model_id', 'gemini-2.5-flash')  # flash for cheaper streaming demo
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION} model={MODEL_ID}')


project=hoehncloudconsole region=us-central1 model=gemini-2.5-flash


## Step 2: Build the real ADK agent with BQ-backed tools

`graph_query` calls BigQuery against the `orgiq` dataset (seeded in notebook 02).
`document_search` queries the RAG corpus from notebook 03. Both make real network calls.
The ADK `InMemoryRunner` drives the full tool-call loop so the AG-UI event generator
just maps ADK event types to their AG-UI equivalents.


In [18]:
import os
from google.cloud import bigquery as _bq
import vertexai
from vertexai import rag as _rag
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# Route ADK / genai SDK through Vertex AI instead of the public Gemini API endpoint.
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION

BQ_DATASET = gcp.get('bq_dataset', 'orgiq')
CORPUS_NAME = gcp.get('rag_corpus_name')

_bq_client = _bq.Client(project=PROJECT_ID)


def graph_query(skill: str, limit: int = 5) -> dict:
    """Find people in the org with a given skill.

    Args:
        skill: skill or domain keyword.
        limit: maximum results.
    """
    query = f"""
        SELECT p.name, p.title, s.name AS skill, ps.weight AS strength
        FROM `{PROJECT_ID}.{BQ_DATASET}.people` p
        JOIN `{PROJECT_ID}.{BQ_DATASET}.person_skills` ps ON ps.person_id = p.id
        JOIN `{PROJECT_ID}.{BQ_DATASET}.skills` s ON s.id = ps.skill_id
        WHERE LOWER(s.name) LIKE LOWER(@pattern)
        ORDER BY ps.weight DESC
    """
    jc = _bq.QueryJobConfig(
        query_parameters=[_bq.ScalarQueryParameter('pattern', 'STRING', f'%{skill}%')]
    )
    rows = list(_bq_client.query(query, job_config=jc).result())[:limit]
    return {
        'results': [
            {'name': r['name'], 'title': r['title'], 'skill': r['skill'], 'strength': float(r['strength'])}
            for r in rows
        ],
        'query_skill': skill,
    }


def document_search(query: str, num_results: int = 3) -> dict:
    """Search org documents via the RAG corpus.

    Args:
        query: natural-language query.
        num_results: top-k chunks.
    """
    if not CORPUS_NAME:
        return {'results': [], 'query': query, 'note': 'no rag_corpus_name in config — run 03-rag-corpus first'}
    try:
        vertexai.init(project=PROJECT_ID, location=REGION)
        resp = _rag.retrieval_query(
            rag_resources=[_rag.RagResource(rag_corpus=CORPUS_NAME)],
            text=query,
            rag_retrieval_config=_rag.RagRetrievalConfig(top_k=num_results),
        )
        contexts = list(resp.contexts.contexts) if hasattr(resp.contexts, 'contexts') else []
        return {
            'results': [
                {'source': getattr(c, 'source_uri', ''), 'text': getattr(c, 'text', '')[:200]}
                for c in contexts
            ],
            'query': query,
        }
    except Exception as e:
        return {'results': [], 'query': query, 'error': str(e)}


orgiq_agent = Agent(
    name='orgiq_agui_agent',
    model=MODEL_ID,
    instruction=(
        'You are OrgIQ Search. Use graph_query to find people with specific skills, '
        'and document_search to find relevant org documents. Cite names and sources.'
    ),
    tools=[graph_query, document_search],
)

print(f'✓ ADK agent: {orgiq_agent.name} / {MODEL_ID}')
print(f'  graph_query     → BigQuery {PROJECT_ID}.{BQ_DATASET}')
print(f'  document_search → RAG corpus {CORPUS_NAME or "(not configured)"}')


✓ ADK agent: orgiq_agui_agent / gemini-2.5-flash
  graph_query     → BigQuery hoehncloudconsole.orgiq
  document_search → RAG corpus projects/hoehncloudconsole/locations/us-central1/ragCorpora/6264524673858404352


In [19]:
import uuid
from typing import AsyncIterator

from ag_ui.core import (
    EventType,
    RunStartedEvent,
    RunFinishedEvent,
    TextMessageStartEvent,
    TextMessageContentEvent,
    TextMessageEndEvent,
    ToolCallStartEvent,
    ToolCallArgsEvent,
    ToolCallEndEvent,
)
from ag_ui.encoder import EventEncoder

_APP_NAME = 'orgiq-agui'
_CHUNK = 40  # characters per TEXT_MESSAGE_CONTENT delta (simulates streaming chunks)


async def orgiq_event_stream(thread_id: str, run_id: str, user_message: str) -> AsyncIterator[object]:
    """Yield the AG-UI event sequence driven by a real ADK + Gemini call."""
    msg_id = f'msg_{uuid.uuid4().hex[:8]}'
    runner = InMemoryRunner(agent=orgiq_agent, app_name=_APP_NAME)
    # create_session is async in current google-adk versions
    session = await runner.session_service.create_session(app_name=_APP_NAME, user_id='agui-demo')

    yield RunStartedEvent(type=EventType.RUN_STARTED, thread_id=thread_id, run_id=run_id)

    _text_started = False

    for evt in runner.run(
        user_id='agui-demo',
        session_id=session.id,
        new_message=genai_types.Content(
            role='user',
            parts=[genai_types.Part(text=user_message)],
        ),
    ):
        if not evt.content or not evt.content.parts:
            continue
        for part in evt.content.parts:
            if getattr(part, 'function_call', None):
                tc_id = f'tool_{uuid.uuid4().hex[:8]}'
                yield ToolCallStartEvent(
                    type=EventType.TOOL_CALL_START,
                    tool_call_id=tc_id,
                    tool_call_name=part.function_call.name,
                    parent_message_id=msg_id,
                )
                yield ToolCallArgsEvent(
                    type=EventType.TOOL_CALL_ARGS,
                    tool_call_id=tc_id,
                    delta=json.dumps(dict(part.function_call.args)),
                )
                yield ToolCallEndEvent(type=EventType.TOOL_CALL_END, tool_call_id=tc_id)

            text = getattr(part, 'text', None)
            if text and evt.author not in ('user',):
                if not _text_started:
                    yield TextMessageStartEvent(
                        type=EventType.TEXT_MESSAGE_START,
                        message_id=msg_id,
                        role='assistant',
                    )
                    _text_started = True
                for i in range(0, len(text), _CHUNK):
                    yield TextMessageContentEvent(
                        type=EventType.TEXT_MESSAGE_CONTENT,
                        message_id=msg_id,
                        delta=text[i:i + _CHUNK],
                    )

    if _text_started:
        yield TextMessageEndEvent(type=EventType.TEXT_MESSAGE_END, message_id=msg_id)
    yield RunFinishedEvent(type=EventType.RUN_FINISHED, thread_id=thread_id, run_id=run_id)


print('✓ orgiq_event_stream defined (real Gemini + BQ tools via ADK InMemoryRunner)')


✓ orgiq_event_stream defined (real Gemini + BQ tools via ADK InMemoryRunner)


## Step 3: Encode to SSE — smoke-test the generator

Run the generator without HTTP first so we can see the wire format and confirm event ordering before the FastAPI hop.


In [20]:
encoder = EventEncoder()

async def _smoke_test():
    captured = []
    async for ev in orgiq_event_stream('thread-001', 'run-001', 'kubernetes experts'):
        captured.append(encoder.encode(ev))
    return captured

events = await _smoke_test()
print(f'✓ emitted {len(events)} SSE chunks\n')
for i, chunk in enumerate(events[:6]):
    print(f'--- chunk {i} ---')
    print(chunk.rstrip())
print(f'\n…({len(events) - 6} more chunks)')


✓ emitted 17 SSE chunks

--- chunk 0 ---
data: {"type":"RUN_STARTED","threadId":"thread-001","runId":"run-001"}
--- chunk 1 ---
data: {"type":"TOOL_CALL_START","toolCallId":"tool_7c143a6c","toolCallName":"graph_query","parentMessageId":"msg_238312f0"}
--- chunk 2 ---
data: {"type":"TOOL_CALL_ARGS","toolCallId":"tool_7c143a6c","delta":"{\"skill\": \"kubernetes\"}"}
--- chunk 3 ---
data: {"type":"TOOL_CALL_END","toolCallId":"tool_7c143a6c"}
--- chunk 4 ---
data: {"type":"TEXT_MESSAGE_START","messageId":"msg_238312f0","role":"assistant"}
--- chunk 5 ---
data: {"type":"TEXT_MESSAGE_CONTENT","messageId":"msg_238312f0","delta":"Here are some Kubernetes experts in the "}

…(11 more chunks)


## Step 4: Wrap the generator in a FastAPI handler

AG-UI uses **POST** for SSE (not the GET-based EventSource), so the body is a `RunAgentInput` JSON. The handler returns a `StreamingResponse` with `media_type='text/event-stream'`. The exact same handler ships to Cloud Run unchanged.


In [22]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

app = FastAPI(title='OrgIQ AG-UI demo')


@app.post('/agent/run')
async def agent_run(request: Request):
    body = await request.json()
    thread_id = body.get('thread_id', f'thread_{uuid.uuid4().hex[:8]}')
    run_id = body.get('run_id', f'run_{uuid.uuid4().hex[:8]}')
    messages = body.get('messages', [])
    user_msg = messages[-1]['content'] if messages else 'who knows kubernetes?'

    enc = EventEncoder()

    async def gen():
        async for ev in orgiq_event_stream(thread_id, run_id, user_msg):
            yield enc.encode(ev)

    return StreamingResponse(gen(), media_type='text/event-stream')


print('✓ FastAPI app defined; route POST /agent/run')


✓ FastAPI app defined; route POST /agent/run


## Step 5: Boot uvicorn in a background thread

Notebook environments don't have an event loop free to host a long-running server. We run uvicorn in a thread on `127.0.0.1:8765` so subsequent cells can hit it like a real client.


In [23]:
import threading, time
import uvicorn

_server: uvicorn.Server | None = globals().get('_server')

if _server is None or not _server.started:
    config_uv = uvicorn.Config(app, host='127.0.0.1', port=8765, log_level='warning')
    _server = uvicorn.Server(config_uv)
    threading.Thread(target=_server.run, daemon=True).start()
    # Bounded wait for the server to bind (cap at ~3s).
    for _ in range(30):
        if _server.started:
            break
        time.sleep(0.1)

print(f'✓ uvicorn started={_server.started} on http://127.0.0.1:8765')


✓ uvicorn started=True on http://127.0.0.1:8765


## Step 6: Stream from the client side

`httpx.stream` is the Python analog of `fetch(...).body.getReader()` — it streams the SSE response incrementally so we can render events as they arrive. Each `data: {...}` line round-trips into a JSON object whose `type` field matches the `EventType` enum.


In [24]:
import httpx

received: list[dict] = []
with httpx.stream(
    'POST',
    'http://127.0.0.1:8765/agent/run',
    json={
        'thread_id': 'thread-demo',
        'run_id': 'run-demo',
        'messages': [{'role': 'user', 'content': 'who knows kubernetes?'}],
    },
    timeout=120,  # real Gemini + BQ call — allow up to 2 min
) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if line.startswith('data:'):
            payload = json.loads(line[len('data:'):].strip())
            received.append(payload)
            print(f'{payload["type"]:24s} {json.dumps({k: v for k, v in payload.items() if k != "type"})[:80]}')

print(f'\n✓ received {len(received)} events')


Task exception was never retrieved
future: <Task finished name='Task-327' coro=<BaseApiClient.aclose() done, defined at /usr/local/lib/python3.12/site-packages/google/genai/_api_client.py:2168> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/google/genai/_api_client.py", line 2173, in aclose
    await self._async_httpx_client.aclose()  # type: ignore[union-attr]
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'


RUN_STARTED              {"threadId": "thread-demo", "runId": "run-demo"}
TOOL_CALL_START          {"toolCallId": "tool_56c7f8a0", "toolCallName": "graph_query", "parentMessageId"
TOOL_CALL_ARGS           {"toolCallId": "tool_56c7f8a0", "delta": "{\"skill\": \"kubernetes\"}"}
TOOL_CALL_END            {"toolCallId": "tool_56c7f8a0"}
TEXT_MESSAGE_START       {"messageId": "msg_a8269552", "role": "assistant"}
TEXT_MESSAGE_CONTENT     {"messageId": "msg_a8269552", "delta": "The following people know about Kubernet
TEXT_MESSAGE_CONTENT     {"messageId": "msg_a8269552", "delta": "es:\n* David Kim (Engineering Manager)\n
TEXT_MESSAGE_CONTENT     {"messageId": "msg_a8269552", "delta": "Alex Chen (Staff Engineer)\n* Maria Sant
TEXT_MESSAGE_CONTENT     {"messageId": "msg_a8269552", "delta": "s (Senior Engineer)\n* Priya Patel (Seni
TEXT_MESSAGE_CONTENT     {"messageId": "msg_a8269552", "delta": "r Engineer)\n* James Wright (Engineer)"}
TEXT_MESSAGE_END         {"messageId": "msg_a8269552"}
RUN_F

## Step 7: Reconstruct the final assistant text

Mirroring the AWS notebook's "render to UI" step: concatenate `TEXT_MESSAGE_CONTENT` deltas keyed by `messageId` to reproduce the visible assistant turn.


In [7]:
buffers: dict[str, str] = {}
for ev in received:
    if ev['type'] == 'TEXT_MESSAGE_CONTENT':
        mid = ev.get('messageId') or ev.get('message_id')
        buffers[mid] = buffers.get(mid, '') + ev.get('delta', '')

for mid, txt in buffers.items():
    print(f'\n[{mid}]')
    print(txt)

print(f'\n✓ reconstructed {len(buffers)} assistant message(s)')



[msg_f7be828f]
Based on the org graph, three engineers stand out for "who knows kubernetes?": Alice Chen, Bob Kumar, and Carol Patel. Alice has the highest composite expertise score (0.91).

✓ reconstructed 1 assistant message(s)


## Step 8: Persist endpoint info

Once the agent moves to Cloud Run, the only field that changes is `endpoint_url`. The middleware uses the same SSE contract regardless of where it points.


In [8]:
gcp['ag_ui'] = {
    'endpoint_url': 'http://127.0.0.1:8765/agent/run',
    'model_id': MODEL_ID,
    'events_emitted': sorted({ev['type'] for ev in received}),
    'transport': 'POST text/event-stream',
}
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated under gcp.ag_ui')


✓ config.json updated under gcp.ag_ui


## Promoting to Cloud Run

When you're ready to move beyond the notebook:

```bash
# Dockerfile entrypoint: uvicorn app:app --host 0.0.0.0 --port 8080
gcloud run deploy orgiq-ag-ui \
    --source . \
    --region $REGION \
    --allow-unauthenticated=false \
    --set-env-vars=GOOGLE_CLOUD_PROJECT=$PROJECT_ID
```

Cloud Run handles SSE natively (no extra config). For tighter latency budgets, Agent Engine offers a hosted runtime that already speaks AG-UI; use it once the agent stabilises.


## Done

Next: `13-a2a-protocol.ipynb` (agent-to-agent messaging, mirroring the AWS A2A notebook).
